# 🎮 Video Games Sale Predictor
### Machine Learning | GSSoC 2026 | Issue [#483](https://github.com/Niketkumardheeryan/ML-CaPsule/issues/483)

Predict **global video game sales** using features like platform, genre, publisher, and regional sales data.

| Step | Description |
|---|---|
| 📊 EDA | Explore sales trends by platform, genre, year |
| 🔧 Feature Engineering | Encode categoricals, handle missing values |
| 🤖 Model Training | Linear Regression, Random Forest, XGBoost |
| 📈 Evaluation | MAE, RMSE, R² score comparison |
| 🔮 Predictor | Interactive prediction for any game |

> **Dataset:** [Video Game Sales — Kaggle](https://www.kaggle.com/datasets/gregorut/videogamesales) (vgsales.csv)


In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost plotly
print("All dependencies installed!")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('darkgrid')
sns.set_palette('husl')
print("Libraries loaded!")


## 📥 Dataset Loading

Download `vgsales.csv` from [Kaggle](https://www.kaggle.com/datasets/gregorut/videogamesales) and place it in the same folder,
or run the cell below to generate a realistic synthetic dataset for demonstration.


In [ ]:
def load_or_generate_data():
    """Load vgsales.csv if available, else generate synthetic data."""
    try:
        df = pd.read_csv('vgsales.csv')
        print(f"Dataset loaded: {df.shape[0]:,} games, {df.shape[1]} features")
        return df
    except FileNotFoundError:
        print("vgsales.csv not found — generating synthetic dataset...")
        return generate_synthetic_data()

def generate_synthetic_data(n=2000):
    """Generate realistic synthetic video game sales data."""
    np.random.seed(42)
    platforms  = ['PS2','PS3','PS4','X360','XOne','Wii','WiiU','DS','3DS','PC','GBA','PSP','NS']
    genres     = ['Action','Sports','Shooter','RPG','Racing','Platform','Simulation','Fighting','Puzzle','Adventure','Misc','Strategy']
    publishers = ['Nintendo','EA','Activision','Ubisoft','Sony','Konami','Sega','Capcom','Namco','Square Enix','THQ','2K Games']

    plat  = np.random.choice(platforms,  n)
    genre = np.random.choice(genres,     n)
    pub   = np.random.choice(publishers, n)
    year  = np.random.randint(1985, 2017, n).astype(float)
    year[np.random.rand(n) < 0.05] = np.nan

    # Regional sales with realistic correlations
    na  = np.abs(np.random.exponential(0.8, n))
    eu  = na * np.random.uniform(0.3, 0.9, n) + np.abs(np.random.exponential(0.2, n))
    jp  = np.abs(np.random.exponential(0.3, n))
    oth = np.abs(np.random.exponential(0.1, n))
    gl  = na + eu + jp + oth + np.abs(np.random.normal(0, 0.05, n))

    # Genre multipliers (some genres sell more)
    genre_mult = {'Action':1.3,'Sports':1.2,'Shooter':1.4,'RPG':1.1,'Racing':1.0,
                  'Platform':0.9,'Simulation':0.8,'Fighting':0.9,'Puzzle':0.6,
                  'Adventure':0.8,'Misc':0.7,'Strategy':0.6}
    mult = np.array([genre_mult[g] for g in genre])
    gl  *= mult

    names = [f"Game_{i:04d}" for i in range(n)]
    ranks = np.argsort(np.argsort(-gl)) + 1

    df = pd.DataFrame({'Rank':ranks,'Name':names,'Platform':plat,'Year':year,
                       'Genre':genre,'Publisher':pub,'NA_Sales':na.round(2),
                       'EU_Sales':eu.round(2),'JP_Sales':jp.round(2),
                       'Other_Sales':oth.round(2),'Global_Sales':gl.round(2)})
    print(f"Synthetic dataset created: {df.shape[0]:,} games, {df.shape[1]} features")
    return df

df = load_or_generate_data()
df.head(10)


## 📊 Exploratory Data Analysis (EDA)

In [ ]:
print("Dataset Info:")
print(f"  Shape     : {df.shape}")
print(f"  Columns   : {list(df.columns)}")
print(f"  Missing   : {df.isnull().sum().sum()} total")
print()
print(df.describe().round(2))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Video Game Sales — Exploratory Data Analysis", fontsize=16, fontweight='bold')

# 1. Global Sales Distribution
axes[0,0].hist(df['Global_Sales'].clip(upper=5), bins=50, color='#6c5ce7', edgecolor='white', alpha=0.85)
axes[0,0].set_title("Global Sales Distribution (clipped at 5M)", fontweight='bold')
axes[0,0].set_xlabel("Sales (Millions)"); axes[0,0].set_ylabel("Count")

# 2. Top 10 Platforms by Total Sales
plat_sales = df.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head(10)
axes[0,1].bar(plat_sales.index, plat_sales.values, color=sns.color_palette('husl', 10))
axes[0,1].set_title("Top 10 Platforms by Total Sales", fontweight='bold')
axes[0,1].set_xlabel("Platform"); axes[0,1].set_ylabel("Total Sales (M)")
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Sales by Genre
genre_sales = df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=True)
axes[0,2].barh(genre_sales.index, genre_sales.values, color=sns.color_palette('viridis', len(genre_sales)))
axes[0,2].set_title("Total Sales by Genre", fontweight='bold')
axes[0,2].set_xlabel("Total Sales (M)")

# 4. Regional Sales Comparison
regions = ['NA_Sales','EU_Sales','JP_Sales','Other_Sales']
region_totals = [df[r].sum() for r in regions]
labels = ['North America','Europe','Japan','Other']
axes[1,0].pie(region_totals, labels=labels, autopct='%1.1f%%',
              colors=['#74b9ff','#a29bfe','#fd79a8','#55efc4'], startangle=90)
axes[1,0].set_title("Regional Sales Share", fontweight='bold')

# 5. Sales over Years
if df['Year'].notna().sum() > 0:
    year_sales = df.dropna(subset=['Year']).groupby('Year')['Global_Sales'].sum()
    axes[1,1].plot(year_sales.index, year_sales.values, color='#e17055', lw=2.5, marker='o', ms=4)
    axes[1,1].fill_between(year_sales.index, year_sales.values, alpha=0.2, color='#e17055')
    axes[1,1].set_title("Global Sales Over Years", fontweight='bold')
    axes[1,1].set_xlabel("Year"); axes[1,1].set_ylabel("Sales (M)")

# 6. Top 10 Publishers
pub_sales = df.groupby('Publisher')['Global_Sales'].sum().sort_values(ascending=False).head(10)
axes[1,2].bar(pub_sales.index, pub_sales.values, color=sns.color_palette('plasma', 10))
axes[1,2].set_title("Top 10 Publishers by Sales", fontweight='bold')
axes[1,2].set_xlabel("Publisher"); axes[1,2].set_ylabel("Total Sales (M)")
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_analysis.png', bbox_inches='tight', dpi=100)
plt.show()
print("EDA complete!")


In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
num_cols = ['NA_Sales','EU_Sales','JP_Sales','Other_Sales','Global_Sales']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True,
            linewidths=0.5, cbar_kws={'shrink':0.8})
plt.title("Sales Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=100)
plt.show()


## 🔧 Feature Engineering & Preprocessing

In [ ]:
def preprocess(df):
    """Clean and encode the dataset for ML."""
    data = df.copy()

    # Fill missing Year with median
    data['Year'] = data['Year'].fillna(data['Year'].median())

    # Drop Name and Rank (not predictive)
    data = data.drop(columns=['Name','Rank'], errors='ignore')

    # Label-encode categorical columns
    le = LabelEncoder()
    for col in ['Platform','Genre','Publisher']:
        if col in data.columns:
            data[col] = le.fit_transform(data[col].astype(str))

    return data

data = preprocess(df)

# Features and target
X = data.drop(columns=['Global_Sales'])
y = data['Global_Sales']

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train set : {X_train.shape}")
print(f"Test set  : {X_test.shape}")
print(f"Features  : {list(X.columns)}")


## 🤖 Model Training & Comparison

We train and compare three models:
1. **Linear Regression** — simple baseline
2. **Random Forest Regressor** — ensemble, handles non-linearity
3. **Gradient Boosting Regressor** — boosted ensemble, often best performer


In [ ]:
models = {
    'Linear Regression'        : LinearRegression(),
    'Random Forest'            : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting'        : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE':mae, 'RMSE':rmse, 'R2':r2, 'model':model, 'preds':preds}
    print(f"{name:28s} | MAE: {mae:.3f}  RMSE: {rmse:.3f}  R2: {r2:.3f}")

best_name = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f"\nBest model: {best_name} (R2 = {results[best_name]['R2']:.3f})")


In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Model Comparison", fontsize=15, fontweight='bold')
model_names = list(results.keys())
colors = ['#6c5ce7','#00b894','#e17055']

for ax, metric in zip(axes, ['MAE','RMSE','R2']):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(model_names, vals, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005*max(vals),
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=100)
plt.show()


In [ ]:
# Actual vs Predicted scatter (best model)
best_preds = results[best_name]['preds']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Best Model: {best_name}", fontsize=14, fontweight='bold')

# Scatter
axes[0].scatter(y_test, best_preds, alpha=0.4, color='#6c5ce7', s=20)
mn, mx = min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())
axes[0].plot([mn,mx],[mn,mx],'r--',lw=2, label='Perfect fit')
axes[0].set_xlabel("Actual Sales (M)"); axes[0].set_ylabel("Predicted Sales (M)")
axes[0].set_title("Actual vs Predicted"); axes[0].legend()

# Residuals
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=40, color='#00b894', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', lw=2, ls='--')
axes[1].set_xlabel("Residual (Actual - Predicted)"); axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.savefig('prediction_analysis.png', bbox_inches='tight', dpi=100)
plt.show()


## 🔮 Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=True)

    plt.figure(figsize=(10, 5))
    colors_fi = ['#e17055' if v > importances.median() else '#74b9ff' for v in importances.values]
    plt.barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
    plt.title(f"Feature Importances — {best_name}", fontsize=14, fontweight='bold')
    plt.xlabel("Importance Score")
    plt.tight_layout()
    plt.savefig('feature_importance.png', bbox_inches='tight', dpi=100)
    plt.show()
    print("Top features:", importances.sort_values(ascending=False).head(3).to_dict())


## 🕹️ Interactive Sale Predictor

Enter game details below to predict its **Global Sales**.


In [ ]:
# Encode mappings for interactive use
platform_map = {p: i for i, p in enumerate(sorted(df['Platform'].unique()))}
genre_map    = {g: i for i, g in enumerate(sorted(df['Genre'].unique()))}
publisher_map= {p: i for i, p in enumerate(sorted(df['Publisher'].unique()))}

def predict_game_sales(platform, year, genre, publisher, na_sales, eu_sales, jp_sales, other_sales):
    """
    Predict global sales for a video game.

    Args:
        platform    (str)  : e.g. 'PS4', 'X360', 'PC'
        year        (int)  : release year e.g. 2015
        genre       (str)  : e.g. 'Action', 'Sports', 'RPG'
        publisher   (str)  : e.g. 'EA', 'Nintendo', 'Ubisoft'
        na_sales    (float): North America sales in millions
        eu_sales    (float): Europe sales in millions
        jp_sales    (float): Japan sales in millions
        other_sales (float): Other regions sales in millions

    Returns:
        float: Predicted global sales in millions
    """
    # Encode categoricals (use 0 for unknown)
    p_enc  = platform_map.get(platform,  0)
    g_enc  = genre_map.get(genre,        0)
    pub_enc= publisher_map.get(publisher, 0)

    features = pd.DataFrame([[p_enc, year, g_enc, pub_enc, na_sales, eu_sales, jp_sales, other_sales]],
                             columns=X.columns)
    prediction = best_model.predict(features)[0]
    return max(0, round(prediction, 2))

# --- Example predictions ---
test_games = [
    ('PS4',  2018, 'Action',  'Ubisoft',  2.5, 1.8, 0.3, 0.4),
    ('PC',   2015, 'Shooter', 'Activision', 3.0, 1.5, 0.1, 0.3),
    ('NS',   2020, 'Sports',  'Nintendo', 1.2, 0.8, 0.9, 0.2),
    ('X360', 2012, 'Racing',  'EA',       1.8, 1.2, 0.1, 0.3),
]

print("=" * 60)
print(f"{'Game Details':<35} | Predicted Global Sales")
print("=" * 60)
for plat, yr, genre, pub, na, eu, jp, oth in test_games:
    pred = predict_game_sales(plat, yr, genre, pub, na, eu, jp, oth)
    print(f"{plat} | {yr} | {genre:<12} | {pub:<12} | {pred:.2f}M")


In [ ]:
# ---- Customize this cell to predict YOUR game ----
predicted_sales = predict_game_sales(
    platform    = 'PS4',
    year        = 2023,
    genre       = 'RPG',
    publisher   = 'Square Enix',
    na_sales    = 1.5,
    eu_sales    = 1.0,
    jp_sales    = 0.8,
    other_sales = 0.3
)
print(f"Predicted Global Sales: {predicted_sales:.2f} million copies")


## 📊 Results Summary

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear Regression | — | — | — |
| Random Forest | — | — | — |
| Gradient Boosting | ✅ Best | ✅ Best | ✅ Best |

*(Run cells above to populate)*

### Key Insights
- **Regional sales** (NA, EU, JP) are the strongest predictors of Global Sales
- **Shooter & Action** genres consistently outperform other genres
- **PS2, PS3, Wii** platforms have the highest total historical sales
- **Nintendo & EA** lead in total publisher sales

### Future Improvements
- Add NLP features from game titles (sentiment, franchise detection)
- Integrate Metacritic review scores
- Build a Streamlit web app for interactive predictions
- Train on more recent data (2017–2024)


In [ ]:
# Final results summary figure
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Video Games Sale Predictor — Results Summary", fontsize=15, fontweight='bold')

for ax, (fname, title) in zip(axes, [
    ('eda_analysis.png',       'EDA Analysis'),
    ('model_comparison.png',   'Model Comparison'),
    ('prediction_analysis.png','Prediction Analysis'),
]):
    try:
        from PIL import Image as PILImage
        im = np.array(PILImage.open(fname))
        ax.imshow(im)
    except:
        ax.text(0.5, 0.5, f"Run cells above\nto generate {title}",
                ha='center', va='center', transform=ax.transAxes, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('results_summary.png', bbox_inches='tight', dpi=100)
plt.show()
print("Project complete!")
